# Lander Videos

Elise flies by feel.

This notebook records and displays videos for the preserved 253-score 10D SolarSystemLander checkpoint.

## Set up

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

In [ ]:
# cell: video-setup # requires: colab-setup

import json
from pathlib import Path

import torch

from dqn.model import DQN
from hpo.evaluation.video import record_video
from hpo.solar_system_lander.environment import DEFAULT_WORLD_MIX, EnvFactory, World

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "10d"
STUDY_SERIES_NAME = f"solar_system_lander_{OBSERVATION_MODE}_elise_stp"
CHECKPOINT_DIR = COLAB.drive_study_dir / "best_checkpoints" / STUDY_SERIES_NAME
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.pt"
CHECKPOINT_METADATA_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.json"
VIDEO_DIR = COLAB.drive_study_dir / "videos" / STUDY_SERIES_NAME

ENV_FACTORY = EnvFactory(OBSERVATION_MODE, world_mix=DEFAULT_WORLD_MIX)

metadata = json.loads(CHECKPOINT_METADATA_PATH.read_text())
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"score: {metadata['score']:.1f}")
print(f"trial: {metadata['trial_number']}")
metadata["world_scores"]

## Record videos

In [ ]:
WORLDS = [
    World.MERCURY,
    World.VENUS,
    World.EARTH,
    World.MOON,
    World.MARS,
]
SEEDS = [7]  # [7, 42, 1911, 2011, 4711]

In [ ]:
# cell: record-videos # requires: video-setup

from hpo.evaluation.rendering.solar_system_lander import render_config

video_paths = [
    record_video(
        CHECKPOINT_PATH,
        DQN,
        ENV_FACTORY.make_env(world, render_mode="rgb_array"),
        seed=seed,
        output_dir=VIDEO_DIR,
        device=device,
        render_cfg=render_config([world], overlay=True),
    )
    for world in WORLDS
    for seed in SEEDS
]

video_paths

In [ ]:
# cell: record-videos-detailed-eagle # requires: video-setup

from hpo.evaluation.rendering.solar_system_lander import render_config

video_paths = [
    record_video(
        CHECKPOINT_PATH,
        DQN,
        ENV_FACTORY.make_env(world, render_mode="rgb_array"),
        seed=seed,
        output_dir=VIDEO_DIR / "detailed_eagle",
        device=device,
        render_cfg=render_config([world], overlay=True, skin="detailed_eagle"),
    )
    for world in WORLDS
    for seed in SEEDS
]

video_paths

## Play videos

In [ ]:
# cell: inspect-video-conditions # requires: objective-config, record-videos

from hpo.evaluation.video import display_video, show_video_conditions

show_video_conditions(ENV_FACTORY, WORLDS, SEEDS)


In [ ]:
display_video(video_paths, 1)